# Part 1 · Climate Data Storytelling with Time Series

<br>

## Tutorial Overview

This tutorial introduces a practical workflow for turning climate data into clear, compelling stories. You will work with Berkeley Earth temperature records, analyze long-term trends, and create visualizations that communicate change, context, and significance.

<br>

<table>
<tr><td width="140"><b>Duration</b></td><td>~1 hours</td></tr>
<tr><td><b>Level</b></td><td>Beginner to Intermediate</td></tr>
<tr><td><b>Prerequisites</b></td><td>Basic Python, pandas, and familiarity with data tables</td></tr>
<tr><td><b>Dataset</b></td><td>Berkeley Earth Surface Temperature (233 countries, 1743–2020)</td></tr>
</table>

<br>

## Learning Objectives

By the end of this tutorial, you will be able to:

- Load and explore real climate data with **pandas**
- Explain the concept of **temperature anomaly**
- Analyze long-term warming patterns using **time series methods**
- Create effective visualizations with **Altair**
- Build a simple **data story** from trends, comparisons, and context


<br>

---

## Step 1 · Understand the Data and the Storytelling Goal

In this step, you will learn what data storytelling is, what this dataset contains, and why temperature anomaly is useful for climate analysis.

<br>

### 1.1 What Is Data Storytelling?

Data storytelling combines three elements:

<table>
<tr><td width="140"><b>Data</b></td><td>Accurate, relevant information</td></tr>
<tr><td><b>Visualization</b></td><td>Clear, interpretable charts</td></tr>
<tr><td><b>Narrative</b></td><td>Context, meaning, and implications</td></tr>
</table>

<br>

> The goal is not simply to display data, but to help an audience understand why the pattern matters.

<br>

### 1.2 Dataset: Berkeley Earth

Berkeley Earth compiles historical temperature records across countries and years, making it useful for long-term climate analysis.

<br>

### 1.3 Data Schema

<table>
<tr><td width="180"><code>Years</code></td><td>Year of observation (1743–2020)</td></tr>
<tr><td><code>Month</code></td><td>Month (1–12)</td></tr>
<tr><td><code>Country</code></td><td>Country name</td></tr>
<tr><td><code>Temperature</code></td><td>Absolute temperature (°C)</td></tr>
<tr><td><code>Anomaly</code></td><td>Temperature anomaly (°C) relative to 1951-1980 country/month average (Berkeley Earth standard baseline)</td></tr>
</table>

<br>

### 1.4 Why Temperature Anomaly?

**Temperature anomaly** measures the difference from a baseline average.

Why it is useful:
- It supports comparison across regions
- It reduces the influence of seasonal differences
- It emphasizes **change** over time
- It aligns with common climate science practice

```text
Baseline average (1951–1980): 14.0°C
Observed temperature:         15.2°C
------------------------------------
Anomaly: +1.2°C
```

👉 Reflect: Why might anomaly be more useful than absolute temperature when comparing multiple countries?


<br>

---

## Step 2 · Set Up the Environment

In this step, you will install and import the libraries used throughout the tutorial.

<br>

### 2.1 Install Required Packages

Run this in your terminal:

```bash
python -m venv venv
source venv/bin/activate  # Windows: venv\\Scripts\\activate

pip install pandas numpy altair streamlit jupyter scipy
```

### 2.2 Import Libraries


In [ ]:
import os
import pandas as pd
import numpy as np
import altair as alt
from scipy import stats

# Enable larger datasets in Altair
alt.data_transformers.disable_max_rows()

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print('Libraries imported successfully')
print(f'  pandas: {pd.__version__}')
print(f'  numpy: {np.__version__}')
print(f'  altair: {alt.__version__}')

<br>

---

## Step 3 · Load and Inspect the Dataset

In this step, you will load the Berkeley Earth dataset and inspect its structure.

<br>

### 3.1 Load the Dataset


In [ ]:
# Load the Berkeley Earth dataset
df = pd.read_csv('data/ddbb_surface_temperature_countries.csv')

print(f'Dataset Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Columns: {df.columns.tolist()}')

In [ ]:
# Preview the data
df.head(10)

### 3.2 Basic Exploration


In [ ]:
print('Data Types:')
print(df.dtypes)
print('\n' + '=' * 50)

print('\nSummary Statistics:')
df.describe()

In [ ]:
print('Missing Values:')
missing = df.isnull().sum()
print(missing)

print(f"\nYear range: {df['Years'].min()} - {df['Years'].max()}")
print(f"Number of countries: {df['Country'].nunique()}")

### 3.3 Explore Available Countries


In [ ]:
countries = sorted(df['Country'].unique())
print(f'Total countries: {len(countries)}\n')

print('Sample countries:')
for i in range(0, min(30, len(countries)), 3):
    row = countries[i:i+3]
    print(f"  {row[0]:<25} {row[1] if len(row) > 1 else '':<25} {row[2] if len(row) > 2 else ''}")


### 3.4 Clean the Data

For time series analysis, we remove rows with missing anomaly values and create a modern-era subset.


In [ ]:
df_clean = df.dropna(subset=['Anomaly']).copy()
print(f"Clean dataset: {len(df_clean):,} rows (removed {len(df) - len(df_clean):,} rows with missing anomaly values)")

👉 Try: Inspect how many rows were removed and think about whether missingness could affect interpretation.


<br>

---

## Step 4 · Build a Country-Level Time Series

In this step, you will select one country and convert monthly observations into annual summaries.

<br>

### 4.1 Select a Country


In [ ]:
# Change this to explore different countries
country = 'South Korea'

country_data = df_clean[df_clean['Country'] == country].copy()

print(f'Selected Country: {country}')
print(f"Data Range: {country_data['Years'].min()} - {country_data['Years'].max()}")
print(f'Total Monthly Records: {len(country_data):,}')

👉 Try changing the `country` value to compare different regions.


### 4.2 Calculate Annual Averages


In [ ]:
annual = country_data.groupby('Years').agg({
    'Anomaly': 'mean',
    'Temperature': 'mean'
}).reset_index()

annual.columns = ['Year', 'Anomaly', 'Temperature']

print(f'Annual Records: {len(annual)}')
print('\nLast 10 years:')
annual.tail(10)

### 4.3 Calculate Decade Averages


In [ ]:
annual['Decade'] = (annual['Year'] // 10) * 10

decade_avg = annual.groupby('Decade')['Anomaly'].mean().reset_index()
decade_avg.columns = ['Decade', 'Avg_Anomaly']

print(f'Decade Averages for {country}:\n')
print('Decade    │ Avg Anomaly │ Visual')
print('──────────┼─────────────┼' + '─' * 30)

for _, row in decade_avg[decade_avg['Decade'] >= 1900].iterrows():
    bar_length = int(abs(row['Avg_Anomaly']) * 15)
    if row['Avg_Anomaly'] >= 0:
        bar = '▓' * bar_length
    else:
        bar = '░' * bar_length
    print(f"{int(row['Decade'])}s     │ {row['Avg_Anomaly']:+.3f}°C    │ {bar}")

### 4.4 Calculate Moving Averages

Moving averages help reveal longer-term patterns by smoothing short-term variation.


In [ ]:
annual['MA_5'] = annual['Anomaly'].rolling(window=5, center=True).mean()
annual['MA_10'] = annual['Anomaly'].rolling(window=10, center=True).mean()

print('Annual data with moving averages (last 15 years):')
annual[['Year', 'Anomaly', 'MA_5', 'MA_10']].tail(15)

👉 Observe: How does the 10-year moving average change the appearance of the trend compared with the raw annual line?


<br>

---

## Step 5 · Identify Story Angles in the Data

In this step, you will move from analysis to interpretation by identifying trend, extremes, and comparisons.

<br>

### 5.1 Questions That Help Build a Story

<table>
<tr><td width="220"><b>What is the overall trend?</b></td><td>Rising, falling, or stable?</td></tr>
<tr><td><b>When did major changes occur?</b></td><td>Possible inflection points</td></tr>
<tr><td><b>Are there notable extremes?</b></td><td>Record highs and lows</td></tr>
<tr><td><b>How does this compare?</b></td><td>Across countries or decades</td></tr>
<tr><td><b>Why does it matter?</b></td><td>Real-world significance</td></tr>
</table>

<br>

### 5.2 Hottest and Coldest Years


In [ ]:
print(f'TOP 10 HOTTEST YEARS in {country}')
print('=' * 40)
hottest = annual.nlargest(10, 'Anomaly')[['Year', 'Anomaly']]
for rank, (_, row) in enumerate(hottest.iterrows(), 1):
    bar = '█' * int(max(row['Anomaly'], 0) * 10)
    print(f"  {rank:2}. {int(row['Year'])}: {row['Anomaly']:+.2f}°C  {bar}")

In [ ]:
print(f'TOP 10 COLDEST YEARS in {country}')
print('=' * 40)
coldest = annual.nsmallest(10, 'Anomaly')[['Year', 'Anomaly']]
for rank, (_, row) in enumerate(coldest.iterrows(), 1):
    bar = '░' * int(abs(row['Anomaly']) * 10)
    print(f"  {rank:2}. {int(row['Year'])}: {row['Anomaly']:+.2f}°C  {bar}")

### 5.3 Estimate a Linear Trend


In [ ]:
modern_data = annual[annual['Year'] >= 1900].dropna(subset=['Anomaly']).copy()

slope, intercept, r_value, p_value, std_err = stats.linregress(
    modern_data['Year'],
    modern_data['Anomaly']
)

print(f'TREND ANALYSIS: {country} (1900–present)')
print('=' * 50)
print(f'  Warming rate: {slope * 100:.3f}°C per century')
print(f'  R² value: {r_value**2:.4f} ({r_value**2 * 100:.1f}% variance explained)')
print(f'  Statistical significance: p = {p_value:.2e}')

if p_value < 0.05:
    print('\n  → Trend is statistically significant (p < 0.05)')
else:
    print('\n  → Trend is not statistically significant (p >= 0.05)')

### 5.4 Compare Historical Periods


In [ ]:
early_period = annual[(annual['Year'] >= 1900) & (annual['Year'] <= 1950)]
mid_period = annual[(annual['Year'] >= 1951) & (annual['Year'] <= 1980)]
recent_period = annual[(annual['Year'] >= 1990) & (annual['Year'] <= 2020)]

early_avg = early_period['Anomaly'].mean()
mid_avg = mid_period['Anomaly'].mean()
recent_avg = recent_period['Anomaly'].mean()

print(f'PERIOD COMPARISON: {country}')
print('=' * 50)
print(f'  Early (1900–1950):  {early_avg:+.3f}°C')
print(f'  Middle (1951–1980): {mid_avg:+.3f}°C  (baseline period)')
print(f'  Recent (1990–2020): {recent_avg:+.3f}°C')
print(f'\n  Change (Early → Recent): {recent_avg - early_avg:+.3f}°C')
print(f'  Change (Middle → Recent): {recent_avg - mid_avg:+.3f}°C')

👉 Reflect: Which result is most useful for a story aimed at a general audience: the hottest years, the linear trend, or the period comparison?


<br>

---

## Step 6 · Visualize the Trend with Altair

In this step, you will create increasingly informative charts, starting with a simple line chart and then adding context.

<br>

### 6.1 Altair Basics

Altair uses a declarative grammar:

```python
alt.Chart(data)
   .mark_*()
   .encode()
   .properties()
```

### 6.2 Common Data Types in Altair

<table>
<tr><td width="80"><code>Q</code></td><td width="120">Quantitative</td><td>Numbers</td></tr>
<tr><td><code>N</code></td><td>Nominal</td><td>Categories</td></tr>
<tr><td><code>O</code></td><td>Ordinal</td><td>Ordered categories</td></tr>
<tr><td><code>T</code></td><td>Temporal</td><td>Dates and times</td></tr>
</table>

<br>

### 6.3 Basic Line Chart


In [ ]:
annual_modern = annual[annual['Year'] >= 1900].copy()

basic_chart = alt.Chart(annual_modern).mark_line(
    color='steelblue',
    strokeWidth=1.5
).encode(
    x=alt.X('Year:Q', title='Year'),
    y=alt.Y('Anomaly:Q', title='Temperature Anomaly (°C)'),
    tooltip=['Year:Q', alt.Tooltip('Anomaly:Q', format='.2f')]
).properties(
    width=700,
    height=400,
    title=f'Temperature Anomaly: {country} (1900–2020)'
).interactive()

basic_chart

### 6.4 Add Reference Lines and Context


In [ ]:
main_line = alt.Chart(annual_modern).mark_line(
    color='steelblue',
    strokeWidth=1.5
).encode(
    x=alt.X('Year:Q', title='Year', scale=alt.Scale(domain=[1900, 2025])),
    y=alt.Y('Anomaly:Q', title='Temperature Anomaly (°C)', scale=alt.Scale(domain=[-2, 3])),
    tooltip=['Year:Q', alt.Tooltip('Anomaly:Q', format='.2f', title='Anomaly')]
)

baseline = alt.Chart(pd.DataFrame({'y': [0]})).mark_rule(
    color='gray',
    strokeDash=[5, 5],
    strokeWidth=1
).encode(y='y:Q')

paris_line = alt.Chart(pd.DataFrame({'y': [1.5]})).mark_rule(
    color='red',
    strokeWidth=2
).encode(y='y:Q')

paris_label = alt.Chart(pd.DataFrame({
    'x': [2022],
    'y': [1.6],
    'text': ['Paris 1.5°C Target']
})).mark_text(
    align='right',
    baseline='bottom',
    color='red',
    fontSize=11,
    fontWeight='bold'
).encode(
    x='x:Q',
    y='y:Q',
    text='text:N'
)

trend_chart = (main_line + baseline + paris_line + paris_label).properties(
    width=700,
    height=400,
    title=f'{country}: Temperature Anomaly with Paris Agreement Target'
).interactive()

trend_chart

### 6.5 Area Chart: Warming vs Cooling Years


In [ ]:
positive = annual_modern[annual_modern['Anomaly'] >= 0]
negative = annual_modern[annual_modern['Anomaly'] < 0]

area_positive = alt.Chart(positive).mark_area(
    color='#e74c3c',
    opacity=0.7,
    line={'color': '#c0392b'}
).encode(
    x=alt.X('Year:Q', title='Year'),
    y=alt.Y('Anomaly:Q', title='Anomaly (°C)')
)

area_negative = alt.Chart(negative).mark_area(
    color='#3498db',
    opacity=0.7,
    line={'color': '#2980b9'}
).encode(
    x='Year:Q',
    y='Anomaly:Q'
)

area_chart = (area_positive + area_negative + baseline).properties(
    width=700,
    height=400,
    title=f'{country}: Warming Years (Red) vs Cooling Years (Blue)'
).interactive()

area_chart

👉 Try: Compare the line chart and the area chart. Which better communicates change to a non-technical audience?


<br>

---

## Step 7 · Create Advanced Storytelling Visualizations

In this step, you will build more expressive visualizations for pattern recognition and comparison.

<br>

### 7.1 Warming Stripes

This visualization represents each year as a colored stripe, moving from blue for cooler years to red for warmer years.


In [ ]:
warming_stripes = alt.Chart(annual_modern).mark_rect().encode(
    x=alt.X('Year:O', axis=alt.Axis(labels=False, ticks=False, title=None, domain=False)),
    color=alt.Color(
        'Anomaly:Q',
        scale=alt.Scale(scheme='redblue', reverse=True, domain=[-2, 2]),
        legend=alt.Legend(title='Anomaly (°C)', orient='bottom')
    )
).properties(
    width=800,
    height=120,
    title=f'Warming Stripes: {country} (1900–2020)'
)

warming_stripes

### 7.2 Decade Bar Chart


In [ ]:
decade_data = decade_avg[decade_avg['Decade'] >= 1900].copy()

decade_bars = alt.Chart(decade_data).mark_bar(size=40).encode(
    x=alt.X('Decade:O', title='Decade', axis=alt.Axis(labelAngle=0)),
    y=alt.Y('Avg_Anomaly:Q', title='Average Anomaly (°C)'),
    color=alt.condition(
        alt.datum.Avg_Anomaly > 0,
        alt.value('#e74c3c'),
        alt.value('#3498db')
    ),
    tooltip=[
        alt.Tooltip('Decade:O', title='Decade'),
        alt.Tooltip('Avg_Anomaly:Q', title='Avg Anomaly', format='.3f')
    ]
).properties(
    width=600,
    height=400,
    title=f'{country}: Average Temperature Anomaly by Decade'
)

decade_bars

### 7.3 Monthly Heatmap


In [ ]:
monthly_data = country_data[country_data['Years'] >= 1960].copy()

heatmap = alt.Chart(monthly_data).mark_rect().encode(
    x=alt.X(
        'Years:O',
        title='Year',
        axis=alt.Axis(labelAngle=-45, values=list(range(1960, 2025, 5)))
    ),
    y=alt.Y(
        'Month:O',
        title='Month',
        sort=list(range(1, 13)),
        axis=alt.Axis(
            labelExpr="datum.value == 1 ? 'Jan' : datum.value == 2 ? 'Feb' : datum.value == 3 ? 'Mar' : datum.value == 4 ? 'Apr' : datum.value == 5 ? 'May' : datum.value == 6 ? 'Jun' : datum.value == 7 ? 'Jul' : datum.value == 8 ? 'Aug' : datum.value == 9 ? 'Sep' : datum.value == 10 ? 'Oct' : datum.value == 11 ? 'Nov' : 'Dec'"
        )
    ),
    color=alt.Color(
        'Anomaly:Q',
        scale=alt.Scale(scheme='redblue', reverse=True, domain=[-3, 3]),
        legend=alt.Legend(title='Anomaly (°C)')
    ),
    tooltip=['Years:O', 'Month:O', alt.Tooltip('Anomaly:Q', format='.2f')]
).properties(
    width=800,
    height=280,
    title=f'{country}: Monthly Temperature Anomalies (1960–2020)'
)

heatmap

### 7.4 Multi-Country Comparison


In [ ]:
compare_countries = ['South Korea', 'Japan', 'Germany', 'Brazil']

multi_country = df_clean[df_clean['Country'].isin(compare_countries)].copy()
multi_annual = multi_country.groupby(['Country', 'Years'])['Anomaly'].mean().reset_index()
multi_annual = multi_annual[multi_annual['Years'] >= 1900]

multi_line = alt.Chart(multi_annual).mark_line(strokeWidth=2).encode(
    x=alt.X('Years:Q', title='Year'),
    y=alt.Y('Anomaly:Q', title='Temperature Anomaly (°C)'),
    color=alt.Color('Country:N', legend=alt.Legend(title='Country')),
    tooltip=['Country:N', 'Years:Q', alt.Tooltip('Anomaly:Q', format='.2f')]
).properties(
    width=750,
    height=420,
    title='Multi-Country Comparison of Annual Temperature Anomalies'
).interactive()

# Get the last year's data point for each country
label_data = multi_annual.groupby('Country').apply(lambda x: x.iloc[-1]).reset_index(drop=True)

# Add country name labels at the end of each line
labels = alt.Chart(label_data).mark_text(
    align='left',
    dx=5,
    fontSize=12,
    fontWeight='bold'
).encode(
    x='Years:Q',
    y='Anomaly:Q',
    text='Country:N',
    color=alt.Color('Country:N', legend=None)
)

(multi_line + labels)

👉 Try: Replace one of the countries and see how the comparison changes. Which comparisons create the strongest story?


<br>

---

## Step 8 · Move from Charts to Narrative

In this step, you will turn analytical results into a concise story.

<br>

### 8.1 Storytelling Principles

- Ask meaningful questions before visualizing
- Use the chart type that best fits the message
- Add context with baselines, thresholds, or comparisons
- Structure your story as: **Hook → Evidence → Meaning**

<br>

### 8.2 A Simple Narrative Template

Use the following structure:

1. **Hook**: What pattern stands out? South Korea's temperature has risen sharply since the mid-twentieth century. All ten of the hottest years on record happened after 1990. The most recent decade, the 2020s, is already 1.26 degrees above the baseline. That is a headline. It makes people want to know more.

2. **Evidence**: Which numbers or charts support it? The trend chart with the Paris reference line shows the warming trajectory approaching 1.5 degrees. The warming stripes show the dramatic color shift from blue to red. The decade bars show acceleration, each decade warmer than the last since the 1990s. The multi-country comparison proves it is not just South Korea. Each chart we built is a piece of evidence.

3. **Meaning**: Why should the audience care? Because the warming rate is 1.337 degrees per century, and it is statistically significant with a p-value near zero. Because recent decades are 0.776 degrees warmer than the 1951 to 1980 baseline. Because this connects to the Paris Agreement, to extreme weather, to real impacts on agriculture and daily life.


<br>

### 8.3 Example Reflection Prompts

- What changed over time?
- Which period shows the strongest warming?
- Which visualization communicates the pattern most clearly?
- What would you highlight for a public audience?


<br>

---

## Step 9 · Export Your Outputs

In this step, you will save your analysis outputs for reuse in reports, slides, or dashboards.


In [ ]:
os.makedirs('output', exist_ok=True)

# Save tabular outputs
annual.to_csv(f'output/{country.lower().replace(" ", "_")}_annual_temperature.csv', index=False)
decade_avg.to_csv(f'output/{country.lower().replace(" ", "_")}_decade_temperature.csv', index=False)

# Save charts as HTML
basic_chart.save(f'output/{country.lower().replace(" ", "_")}_basic_chart.html')
trend_chart.save(f'output/{country.lower().replace(" ", "_")}_trend_chart.html')
area_chart.save(f'output/{country.lower().replace(" ", "_")}_area_chart.html')
warming_stripes.save(f'output/{country.lower().replace(" ", "_")}_warming_stripes.html')
decade_bars.save(f'output/{country.lower().replace(" ", "_")}_decade_bars.html')
heatmap.save(f'output/{country.lower().replace(" ", "_")}_monthly_heatmap.html')
multi_line.save('output/multi_country_comparison.html')

print('Files saved to the output/ directory.')

👉 Checkpoint: Open the exported HTML files and review which charts are most presentation-ready.


<br>

---

## Part 1 Summary

### What You Learned

<table>
<tr><td width="180"><b>Data Skills</b></td><td>Loading, cleaning, grouping, summarizing</td></tr>
<tr><td><b>Analytical Skills</b></td><td>Trend analysis, decade comparison, moving averages</td></tr>
<tr><td><b>Visualization Skills</b></td><td>Line charts, area charts, heatmaps, warming stripes</td></tr>
<tr><td><b>Storytelling Skills</b></td><td>Turning patterns into a clear narrative</td></tr>
</table>

<br>

### Key Takeaways

#### Storytelling Principles

- Ask meaningful questions before visualizing
- Use the right chart for the right message
- Add context with baselines and references
- Structure stories as Hook → Evidence → Meaning

#### Climate Insights

- Temperature anomaly measures change from a baseline
- Warming patterns can differ across regions and periods
- Recent decades often show stronger warming trends
- Visualizations help audiences understand long-term change


<br>

---

## Resources

### Data Sources

<table>
<tr><td width="160">Berkeley Earth</td><td><a href="https://berkeleyearth.org/data/">berkeleyearth.org/data</a></td></tr>
<tr><td>NASA GISS</td><td><a href="https://data.giss.nasa.gov/gistemp/">data.giss.nasa.gov/gistemp</a></td></tr>
<tr><td>NOAA Climate</td><td><a href="https://www.climate.gov/">climate.gov</a></td></tr>
</table>

<br>

### Learning Resources

<table>
<tr><td width="180">Altair Documentation</td><td><a href="https://altair-viz.github.io/">altair-viz.github.io</a></td></tr>
<tr><td>Pandas Documentation</td><td><a href="https://pandas.pydata.org/docs/">pandas.pydata.org/docs</a></td></tr>
<tr><td>Warming Stripes</td><td><a href="https://showyourstripes.info/">showyourstripes.info</a></td></tr>
</table>

<br>

### Citation

```text
Berkeley Earth Surface Temperature Study
http://berkeleyearth.org/

Rohde, R. A. and Hausfather, Z.: The Berkeley Earth Land/Ocean Temperature Record,
Earth Syst. Sci. Data, 12, 3469–3479, https://doi.org/10.5194/essd-12-3469-2020, 2020.
```


<br>

---

## Next: Part 2

In **Part 2**, you will turn these analyses into an interactive Streamlit dashboard, connect user inputs to visualizations, and generate dynamic narrative outputs.
